# Multi-Stage Continuous-Flow Manufacturing Process — Factory Output Prediction

This notebook contains the full project code developed across Week 1 to Week 4:
- Week 1: Data loading, exploration, dataset structure understanding
- Week 2: Data cleaning, correlation analysis, baseline Linear Regression model
- Week 3: Time-lag feature engineering, Random Forest model
- Week 4: Model comparison, Stage 2 output modeling, anomaly detection prototype


## Week 1: Load Data and Explore Structure

In [1]:
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_csv('continuous_factory_process.csv', low_memory=False)

print("Shape:", df.shape)
print("Date range:", df['time_stamp'].iloc[0], "to", df['time_stamp'].iloc[-1])
print("Total columns:", len(df.columns))

Shape: (14088, 116)
Date range: 06-03-2019 10:52 to 06-03-2019 14:47
Total columns: 116


In [2]:
# Convert all columns except timestamp to numeric
for col in df.columns[1:]:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Check for missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
print("Columns with missing values:")
print(missing_pct[missing_pct > 0].sort_values(ascending=False))

Columns with missing values:
Series([], dtype: float64)


In [3]:
# Identify key column groups based on notes_on_dataset.txt
stage1_actual = [c for c in df.columns if 'Stage1.Output' in c and 'U.Actual' in c]
stage1_setpt  = [c for c in df.columns if 'Stage1.Output' in c and 'Setpoint' in c]
stage2_actual = [c for c in df.columns if 'Stage2.Output' in c and 'U.Actual' in c]
stage2_setpt  = [c for c in df.columns if 'Stage2.Output' in c and 'Setpoint' in c]

machine_cols   = [c for c in df.columns if any(f'Machine{n}' in c for n in [1,2,3]) and 'RawMaterial' not in c]
combiner_cols  = [c for c in df.columns if 'Combiner' in c]
machine45_cols = [c for c in df.columns if 'Machine4' in c or 'Machine5' in c]

print("Stage 1 output columns:", len(stage1_actual))
print("Stage 2 output columns:", len(stage2_actual))
print("Machine 1-3 process columns:", len(machine_cols))
print("Combiner columns:", len(combiner_cols))
print("Machine 4-5 process columns:", len(machine45_cols))

Stage 1 output columns: 15
Stage 2 output columns: 15
Machine 1-3 process columns: 21
Combiner columns: 3
Machine 4-5 process columns: 14


In [4]:
# Basic descriptive statistics for Stage 1 outputs vs their setpoints
setpoints = df[stage1_setpt].iloc[0]

stats = pd.DataFrame({
    'Measurement': [f'Measurement{i}' for i in range(15)],
    'Mean':     df[stage1_actual].mean().round(3).values,
    'Std Dev':  df[stage1_actual].std().round(3).values,
    'Min':      df[stage1_actual].min().round(3).values,
    'Max':      df[stage1_actual].max().round(3).values,
    'Setpoint': setpoints.values
})
stats['Avg_Error'] = (stats['Mean'] - stats['Setpoint']).abs().round(3)
print(stats.to_string(index=False))

  Measurement   Mean  Std Dev    Min    Max  Setpoint  Avg_Error
 Measurement0 12.897    0.934  0.000 20.880     13.75      0.853
 Measurement1  8.051    6.904 -3.133 19.140     22.74     14.689
 Measurement2 11.357    1.053 -4.928 23.530     13.02      1.663
 Measurement3 21.326    2.110  0.000 26.240     21.88      0.554
 Measurement4 32.876    3.870 -7.689 34.760     32.55      0.326
 Measurement5  0.124    0.570 -0.580  4.960      2.74      2.616
 Measurement6  1.338    1.136 -0.746  7.030      4.25      2.912
 Measurement7  1.100    1.414 -0.760  5.190      2.97      1.870
 Measurement8 19.754    4.787 -0.000 22.460     21.30      1.546
 Measurement9 17.966    4.197 -0.004 20.450     19.52      1.554
Measurement10  7.682    1.085 -0.001 13.070      8.65      0.968
Measurement11  1.493    2.543 -0.000  7.470      6.16      4.667
Measurement12  1.203    0.664 -0.000  3.950      2.02      0.817
Measurement13  2.881    0.941 -1.225  6.910      3.16      0.279
Measurement14  9.941    7

In [5]:
# Check ambient conditions and Machine 1 stability
print("Ambient Humidity range:", df['AmbientConditions.AmbientHumidity.U.Actual'].min(),
      "-", df['AmbientConditions.AmbientHumidity.U.Actual'].max())
print("Ambient Temperature range:", df['AmbientConditions.AmbientTemperature.U.Actual'].min(),
      "-", df['AmbientConditions.AmbientTemperature.U.Actual'].max())

m1_cols = [c for c in df.columns if 'Machine1' in c and 'RawMaterial' not in c]
print(df[m1_cols].describe().round(2))

Ambient Humidity range: 13.84 - 17.24
Ambient Temperature range: 23.02 - 24.43
       Machine1.Zone1Temperature.C.Actual  Machine1.Zone2Temperature.C.Actual  \
count                            14088.00                            14088.00   
mean                                72.01                               72.01   
std                                  0.06                                0.41   
min                                 71.90                               71.30   
25%                                 72.00                               71.60   
50%                                 72.00                               72.00   
75%                                 72.00                               72.40   
max                                 72.50                               72.70   

       Machine1.MotorAmperage.U.Actual  Machine1.MotorRPM.C.Actual  \
count                         14088.00                    14088.00   
mean                             70.33             

## Week 2: Data Cleaning, Timestamp Fix, Correlation Analysis, Baseline Model

In [6]:
# Fix timestamp format (MM-DD-YYYY, not DD-MM-YYYY)
df['time_stamp'] = pd.to_datetime(df['time_stamp'], format='%m-%d-%Y %H:%M')
print("Timestamp dtype after fix:", df['time_stamp'].dtype)
print(df['time_stamp'].head(3))

Timestamp dtype after fix: datetime64[ns]
0   2019-06-03 10:52:00
1   2019-06-03 10:52:00
2   2019-06-03 10:52:00
Name: time_stamp, dtype: datetime64[ns]


In [7]:
# Remove rows with physically impossible negative Stage 1 output values
print("Negative value counts in Stage 1 outputs:")
for col in stage1_actual:
    n = (df[col] < 0).sum()
    print(f"  {col.split('.')[2]}: {n} rows")

df_clean = df.copy()
for col in stage1_actual:
    df_clean.loc[df_clean[col] < 0, col] = np.nan

df_clean_dropped = df_clean.dropna(subset=stage1_actual)

print(f"\nRows before cleaning: {len(df)}")
print(f"Rows after removing negative Stage1 rows: {len(df_clean_dropped)}")
removed = len(df) - len(df_clean_dropped)
print(f"Rows removed: {removed} ({removed/len(df)*100:.2f}%)")

Negative value counts in Stage 1 outputs:
  Measurement0: 0 rows
  Measurement1: 5 rows
  Measurement2: 1 rows
  Measurement3: 0 rows
  Measurement4: 2 rows
  Measurement5: 19 rows
  Measurement6: 10 rows
  Measurement7: 6 rows
  Measurement8: 2 rows
  Measurement9: 3 rows
  Measurement10: 1 rows
  Measurement11: 9 rows
  Measurement12: 7 rows
  Measurement13: 2 rows
  Measurement14: 11 rows

Rows before cleaning: 14088
Rows after removing negative Stage1 rows: 14053
Rows removed: 35 (0.25%)


In [8]:
# Correlation analysis: which machine inputs affect each Stage 1 output?
input_cols = machine_cols + combiner_cols

corr_results = {}
for out_col in stage1_actual:
    mname = out_col.split('.')[2]
    correlations = []
    for inp in input_cols:
        r = df_clean_dropped[inp].corr(df_clean_dropped[out_col])
        correlations.append((inp, round(r, 3)))
    correlations.sort(key=lambda x: abs(x[1]), reverse=True)
    corr_results[mname] = correlations[:3]

for m, top3 in corr_results.items():
    print(f"\n{m}:")
    for inp, r in top3:
        short = inp.replace('AmbientConditions.', '').replace('.C.Actual','').replace('.U.Actual','')
        print(f"  {short}: r={r}")


Measurement0:
  Machine3.MaterialTemperature: r=0.116
  Machine2.MaterialTemperature: r=0.115
  Machine2.MaterialPressure: r=-0.091

Measurement1:
  FirstStage.CombinerOperation.Temperature1: r=-0.64
  Machine1.MotorRPM: r=-0.637
  FirstStage.CombinerOperation.Temperature2: r=-0.612

Measurement2:
  Machine3.MaterialPressure: r=-0.114
  Machine2.MotorAmperage: r=-0.111
  Machine3.MaterialTemperature: r=-0.104

Measurement3:
  Machine1.MaterialTemperature: r=0.105
  FirstStage.CombinerOperation.Temperature3: r=-0.07
  Machine2.MaterialTemperature: r=0.063

Measurement4:
  Machine3.MaterialTemperature: r=0.216
  Machine1.MotorRPM: r=0.215
  Machine3.MotorRPM: r=-0.209

Measurement5:
  Machine1.MaterialPressure: r=-0.146
  Machine3.MaterialTemperature: r=0.108
  Machine1.MotorAmperage: r=-0.095

Measurement6:
  Machine1.MaterialTemperature: r=-0.549
  Machine3.MaterialTemperature: r=-0.5
  Machine3.MotorRPM: r=0.445

Measurement7:
  Machine1.MotorRPM: r=-0.683
  Machine3.MaterialTemperat

In [9]:
# Baseline Linear Regression model for all 15 Stage 1 outputs
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

X = df_clean_dropped[input_cols].fillna(df_clean_dropped[input_cols].mean())
results = []

for out_col in stage1_actual:
    mname = out_col.split('.')[2]
    y = df_clean_dropped[out_col].fillna(df_clean_dropped[out_col].mean())

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    model = LinearRegression()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    r2  = r2_score(y_test, y_pred)

    results.append({
        'Measurement': mname,
        'MAE (mm)': round(mae, 3),
        'R2 Score': round(r2, 3),
        'Model Quality': 'Good (R2>0.5)' if r2 > 0.5 else ('Fair (0.2-0.5)' if r2 > 0.2 else 'Poor (<0.2)')
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

  Measurement  MAE (mm)  R2 Score  Model Quality
 Measurement0     0.199    -0.014    Poor (<0.2)
 Measurement1     2.340     0.758  Good (R2>0.5)
 Measurement2     0.340     0.031    Poor (<0.2)
 Measurement3     0.539     0.018    Poor (<0.2)
 Measurement4     1.117     0.050    Poor (<0.2)
 Measurement5     0.231     0.033    Poor (<0.2)
 Measurement6     0.618     0.399 Fair (0.2-0.5)
 Measurement7     0.597     0.686  Good (R2>0.5)
 Measurement8     2.545     0.151    Poor (<0.2)
 Measurement9     2.204     0.180    Poor (<0.2)
Measurement10     0.313     0.005    Poor (<0.2)
Measurement11     0.563     0.860  Good (R2>0.5)
Measurement12     0.346     0.424 Fair (0.2-0.5)
Measurement13     0.524     0.312 Fair (0.2-0.5)
Measurement14     5.183     0.265 Fair (0.2-0.5)


## Week 3: Time-Lag Feature Engineering and Random Forest Model

In [10]:
# Build a working sample and create 1-2 second lag features for machine/combiner inputs
clean = df.copy()
for c in stage1_actual:
    clean.loc[clean[c] < 0, c] = np.nan
clean = clean.dropna(subset=stage1_actual).iloc[:5000].copy()

for lag in range(1, 3):
    shifted = clean[input_cols].shift(lag)
    shifted.columns = [f'{col}_lag{lag}' for col in input_cols]
    clean = pd.concat([clean, shifted], axis=1)

clean = clean.dropna().copy()
all_features = input_cols + [c for c in clean.columns if c.endswith('_lag1') or c.endswith('_lag2')]

print("Rows used:", len(clean))
print("Total features:", len(all_features))

Rows used: 4998
Total features: 72


In [11]:
# Train Random Forest on selected Stage 1 measurements (that were weak with Linear Regression)
from sklearn.ensemble import RandomForestRegressor

selected = [
    'Stage1.Output.Measurement6.U.Actual',
    'Stage1.Output.Measurement14.U.Actual',
    'Stage1.Output.Measurement11.U.Actual'
]

rf_rows = []
for out in selected:
    X = clean[all_features]
    y = clean[out]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    rf = RandomForestRegressor(n_estimators=40, random_state=42, n_jobs=-1, max_depth=8)
    rf.fit(X_train, y_train)
    pred = rf.predict(X_test)

    imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(3)
    rf_rows.append([
        out.split('.')[2],
        round(mean_absolute_error(y_test, pred), 3),
        round(r2_score(y_test, pred), 3),
        imp.index[0], imp.index[1], imp.index[2]
    ])

rf_res = pd.DataFrame(rf_rows, columns=['Measurement','MAE','R2','Top1','Top2','Top3'])
print(rf_res.to_string(index=False))

  Measurement   MAE    R2                                       Top1                                       Top2                                       Top3
 Measurement6 0.117 0.869 Machine3.MaterialTemperature.U.Actual_lag1      Machine3.MaterialTemperature.U.Actual Machine3.MaterialTemperature.U.Actual_lag2
Measurement14 0.596 0.894      Machine3.MaterialTemperature.U.Actual Machine3.MaterialTemperature.U.Actual_lag1 Machine3.MaterialTemperature.U.Actual_lag2
Measurement11 0.290 0.873 Machine3.MaterialTemperature.U.Actual_lag1 Machine3.MaterialTemperature.U.Actual_lag2      Machine3.MaterialTemperature.U.Actual


## Week 4: Model Comparison, Stage 2 Modeling, Anomaly Detection

In [12]:
# Compare Linear Regression vs Random Forest on the same 3 measurements
comparison = []
for out in selected:
    X = clean[all_features]
    y = clean[out]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    lr = LinearRegression()
    lr.fit(X_train, y_train)
    pred_lr = lr.predict(X_test)
    mae_lr = mean_absolute_error(y_test, pred_lr)
    r2_lr = r2_score(y_test, pred_lr)

    rf = RandomForestRegressor(n_estimators=40, random_state=42, n_jobs=-1, max_depth=8)
    rf.fit(X_train, y_train)
    pred_rf = rf.predict(X_test)
    mae_rf = mean_absolute_error(y_test, pred_rf)
    r2_rf = r2_score(y_test, pred_rf)

    comparison.append({
        'Measurement': out.split('.')[2],
        'LR_MAE': round(mae_lr,3), 'LR_R2': round(r2_lr,3),
        'RF_MAE': round(mae_rf,3), 'RF_R2': round(r2_rf,3)
    })

comp_df = pd.DataFrame(comparison)
print(comp_df.to_string(index=False))

  Measurement  LR_MAE  LR_R2  RF_MAE  RF_R2
 Measurement6   0.553  0.528   0.117  0.869
Measurement14   4.459  0.204   0.596  0.894
Measurement11   0.833  0.724   0.290  0.873


In [13]:
# Stage 2 output modeling using Stage 1 outputs + Machine 4/5 readings as inputs
clean2 = df.copy()
for c in stage2_actual:
    clean2.loc[clean2[c] < 0, c] = np.nan
clean2 = clean2.dropna(subset=stage2_actual).iloc[:5000].copy()

stage2_input_cols = stage1_actual + machine45_cols
clean2 = clean2.dropna(subset=stage2_input_cols).copy()

stage2_selected = [
    'Stage2.Output.Measurement6.U.Actual',
    'Stage2.Output.Measurement14.U.Actual',
    'Stage2.Output.Measurement11.U.Actual'
]

stage2_results = []
for out in stage2_selected:
    X = clean2[stage2_input_cols]
    y = clean2[out]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    rf = RandomForestRegressor(n_estimators=40, random_state=42, n_jobs=-1, max_depth=8)
    rf.fit(X_train, y_train)
    pred = rf.predict(X_test)

    mae = mean_absolute_error(y_test, pred)
    r2 = r2_score(y_test, pred)
    stage2_results.append({'Measurement': out.split('.')[2], 'MAE': round(mae,3), 'R2': round(r2,3)})

s2_df = pd.DataFrame(stage2_results)
print("Rows used for Stage 2:", len(clean2))
print(s2_df.to_string(index=False))

Rows used for Stage 2: 5000
  Measurement   MAE    R2
 Measurement6 0.108 0.461
Measurement14 0.514 0.787
Measurement11 0.163 0.892


In [14]:
# Simple anomaly detection prototype using prediction error (mean + 3*std rule)
best_out = 'Stage1.Output.Measurement11.U.Actual'

X = clean[all_features]
y = clean[best_out]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf = RandomForestRegressor(n_estimators=40, random_state=42, n_jobs=-1, max_depth=8)
rf.fit(X_train, y_train)
pred = rf.predict(X_test)

errors = np.abs(y_test.values - pred)
mean_err = errors.mean()
std_err = errors.std()
threshold = mean_err + 3 * std_err
anomalies = (errors > threshold).sum()

print(f"Mean error: {mean_err:.3f}")
print(f"Std error: {std_err:.3f}")
print(f"Anomaly threshold (mean+3std): {threshold:.3f}")
print(f"Anomalies flagged: {anomalies} out of {len(errors)} test points ({anomalies/len(errors)*100:.2f}%)")

Mean error: 0.290
Std error: 0.884
Anomaly threshold (mean+3std): 2.942
Anomalies flagged: 37 out of 1000 test points (3.70%)


## Summary

| Week | Focus | Key Result |
|---|---|---|
| Week 1 | Data exploration | 14,088 rows, 116 columns mapped |
| Week 2 | Cleaning + Linear Regression | 35 rows cleaned; LR strong on 3/15 measurements |
| Week 3 | Lag features + Random Forest | Big R² gains on Measurement6, 11, 14 |
| Week 4 | Model comparison + Stage 2 + anomaly detection | RF beats LR; Stage 2 partly predictable; basic anomaly flag built |
